In [61]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [62]:
train = pd.read_csv("../data/processed/train_clean.csv")
test = pd.read_csv("../data/processed/test_clean.csv")

In [63]:
train["SpendBin"] = pd.qcut(
    train["Total Spend"],
    q=10,
    labels=False,
    duplicates="drop"
)

test["SpendBin"] = pd.qcut(
    test["Total Spend"],
    q=10,
    labels=False,
    duplicates="drop"
)

In [64]:
categorical_cols = ["Contract Length", "Subscription Type", "Gender"]

numeric_cols = [
    "Total Spend",
    "Support Calls",
    "Usage Frequency",
    "Age",
    "Last Interaction",
    "Tenure",
    "SpendBin", 
    #"Payment Delay",
]

rf_features = categorical_cols + numeric_cols

In [65]:
rf_preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    # (StandardScaler(), numeric_cols),
    # remainder="drop"
    remainder="passthrough"
)

In [66]:
rf_model = RandomForestClassifier(
    # n_estimators=300,                 # number of trees
    # max_depth=20,                    # fairly deep
    # min_samples_leaf=100,            # regularization
    # max_features="sqrt",             # standard RF setting
    # class_weight="balanced_subsample",  # help class imbalance
    # n_jobs=-1,
    # random_state=1234,
    
    max_depth=20, #14,        # prevents overfitting
    min_samples_leaf=100, #200,
    #class_weight="balanced",
    random_state=1234
)

# tree_model = DecisionTreeClassifier(
#     max_depth=10, #14,        # prevents overfitting
#     min_samples_leaf=240, #200,
#     #class_weight="balanced",
#     random_state=1234
# )

In [67]:
rf_pipeline = make_pipeline(
    rf_preprocessor,
    rf_model
)

In [68]:
X = train[rf_features].copy()
y = train["Churn"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1234,
    stratify=y
)

In [69]:
rf_fit = rf_pipeline.fit(X_train, y_train)

val_pred_rf = rf_fit.predict_proba(X_val)[:, 1]
auc_rf = roc_auc_score(y_val, val_pred_rf)
auc_rf


0.925392049942801

In [59]:
X_test = test[rf_features].copy()
test_pred_rf = rf_fit.predict_proba(X_test)[:, 1]

In [60]:
submission_rf = pd.DataFrame({
    "CustomerID": test["CustomerID"],
    "Churn": test_pred_rf
})

submission_rf.to_csv("../data/submissions/random_forest.csv", index=False)